#
RLCT Estimation of Sorting

This Jupyter Notebook aims to measure the Real Log Canonical Threshold (RLCT) for a small 3-layer transformer model (~280,000 parameters) trained to sort sequences of 20 digits consisting of the numbers 0-19. It uses both Stochastic Gradient Nose-Hoover Thermostat (SGNHT) and Stochastic Gradient Langevin Dynamics (SGLD) as sampling methods.

## Main Steps:

1. **Data Preparation**: Generate the dataset of numbers to sort.
2. **Model Training**: Train a transformer model using stochastic gradient descent.
3. **Model Evaluation**: Evaluate the model's performance on a test set.
4. **RLCT Estimation**: Use SGNHT and SGLD samplers to estimate RLCT.
5. **Plotting**: Visualize train and test losses, and RLCT estimates.

In [ ]:
%pip install devinterp seaborn torchvision pickleshare wandb plotly einops scikit-learn
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD
%cd ..

In [ ]:
import numpy as np
import torch as t
import torch
import torch.nn as nn
import torch.optim as optim
import time
import torch.nn.functional as F
import einops
import random
import helpers
from transformers import *
from dataclasses import dataclass
import os
import copy
import wandb
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from python.optim import EntropySGD
from torch.utils.data import DataLoader

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT

PRIMARY, SECONDARY, TERTIARY, QUATERNARY = sns.color_palette("muted")[:4]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT = sns.color_palette(
    "pastel"
)[:4]


In [ ]:
def full_accuracy(config, model, data):
    logits = model(data)[:, -1]
    labels = t.tensor([config.fn(i, j) for i, j, _ in data]).to(config.device)
    return (logits.argmax(1) == labels).float().mean()

def accuracy_function(outputs, targets):
    return (outputs[ : , -1].argmax(1) == targets).float().mean()

def do_a_training_step(config, model, train_data, test_data, optimizer, scheduler, epoch: int):
        '''returns train_loss, test_loss'''
        model.train()
        train_loss = full_loss(config=config, model=model, data=train_data)
        train_accuracy = full_accuracy(config=config, model=model, data=train_data)
        #self.train_losses.append(train_loss.item())
        #self.test_losses.append(test_loss.item())
        train_loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        model.eval()  # Set model to evaluation mode
        with torch.no_grad():  # Disable gradient calculation for test
            test_loss = full_loss(config=config, model=model, data=test_data)
            test_accuracy = full_accuracy(config=config, model=model, data=test_data)
            
        if epoch % 100 == 0:
            # TODO is this ok? this was np.log, and it was barking at me ; i think np.log was being interpreted as a logging module
            print(f'Epoch {epoch}, train loss {t.log(train_loss).item():.4f}, test loss {t.log(test_loss).item():.4f}')

        return train_loss.detach(), test_loss.detach(), train_accuracy.detach(), test_accuracy.detach()

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, model_key):

    model.train()
    train_loss = 0
    train_accuracy = 0
    for index, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(data.to(DEVICE))
        loss = criterion(outputs, targets.to(DEVICE))
        train_loss += loss.detach().item()
        train_accuracy += accuracy_function(outputs, targets.to(DEVICE))
        loss.backward()
        optimizer.step()
        scheduler.step()
        yield (train_loss, train_accuracy)


def evaluate(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    test_accuracy = 0
    with torch.no_grad():
        for index, (data, targets) in enumerate(test_loader):
            outputs = model(data.to(DEVICE))
            loss = criterion(outputs, targets.to(DEVICE))
            test_loss += loss.item()
            test_accuracy += accuracy_function(outputs, targets.to(DEVICE))

    yield (test_loss, test_accuracy)


In [ ]:
# Constants
DEVICE = "cuda" if t.cuda.is_available() else "cpu"
BATCH_SIZE = 16384
LR = 1e-4
N_EPOCHS = 30000
SAVE_EVERY_N_EPOCHS = 100
config = Config()

def get_data(config : Config):
    num_to_generate = config.p
    pairs = [(i, j, num_to_generate) for i in range(num_to_generate) for j in range(num_to_generate)]
    random.seed(config.seed)
    random.shuffle(pairs)
    div = int(config.frac_train*len(pairs))
    labels = [config.fn(i, j) for i, j, _ in pairs]
    pairs = t.tensor(pairs).long()
    labels = t.tensor(labels).long()
    train_data = list(zip(pairs[:div], labels[:div]))
    test_data = list(zip(pairs[div:], labels[div:]))
    return train_data, test_data, pairs, labels

train_data, test_data, all_data, labels = get_data(config = config)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
criterion = helpers.cross_entropy_high_precision
print(len(train_loader))
print(len(test_loader))
print(len(train_data))
print(len(test_data))
print(torch.cuda.is_available())
print(config.device)

In [ ]:
def train_models(config, train_data, test_data, all_data, labels, runs):
    save_every_n_epochs = SAVE_EVERY_N_EPOCHS
    num_save_steps = N_EPOCHS // save_every_n_epochs
    train_losses = torch.zeros(runs, num_save_steps)
    test_losses = torch.zeros(runs, num_save_steps)
    trig_losses = torch.zeros(runs, num_save_steps)
    excluded_losses = torch.zeros(runs, num_save_steps)
    train_accuracies = torch.zeros(runs, num_save_steps)
    test_accuracies = torch.zeros(runs, num_save_steps)
    models_saved = []
    
    fourier_basis = make_fourier_basis(config = config)
    train, _ = gen_train_test(config = config)
    is_train, is_test = config.is_train_is_test(train = train)
    all_data = torch.tensor([(i, j, config.p) for i in range(config.p) for j in range(config.p)]).to(config.device)
    labels = torch.tensor([config.fn(i, j) for i, j, _ in all_data]).to(config.device)
    indices = get_indices_for_key_freqs_calculation(config)
    
    for run in tqdm(range(runs)):
        model = Transformer(config, use_cache=False)
        model.to(config.device)
        print(sum(p.numel() for p in model.parameters()))
        optimizer = optim.AdamW(model.parameters(), lr = config.lr, weight_decay=config.weight_decay, betas=(0.9, 0.98))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step/10, 1)) # TODO make this a config option
        index = 0
        for epoch in tqdm(range(N_EPOCHS)):
            (train_loss, train_accuracy) = next(train_one_epoch(
                model, train_loader, optimizer, scheduler, criterion, 'sgd'
            ))
            (test_loss, test_accuracy) = next(evaluate(model, test_loader, criterion))
            if epoch % 100 == 0:
              print(
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Loss: {train_loss}, Test Loss: {test_loss}", '\n',
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}"
              )
            #if (epoch % 10 == 0 and epoch < 1000) or epoch % 100 == 0:
            if epoch % save_every_n_epochs == 0:
                train_losses[run, index] = train_loss
                test_losses[run, index] = test_loss
                train_accuracies[run, index] = train_accuracy
                test_accuracies[run, index] = test_accuracy
                models_saved += [copy.deepcopy(model)]
                
                with torch.no_grad():  
                    
                    key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
                    #key_freqs = calculate_key_freqs(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis)
                    '''
                    if epoch >= 29000:
                        key_freqs_old = calculate_key_freqs(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis)
                        print(key_freqs)
                        print(key_freqs_old)
                    '''
                    
                    logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
                    trig_losses[run, index], trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
                    excluded_losses[run, index] = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
                
                index += 1

    train_losses_final = train_losses.mean(dim=0)
    test_losses_final = test_losses.mean(dim=0)
    train_accuracies_final = train_accuracies.mean(dim=0)
    test_accuracies_final = test_accuracies.mean(dim=0)
    trig_losses_final = trig_losses.mean(dim=0)
    excluded_losses_final = excluded_losses.mean(dim=0)
    torch.cuda.empty_cache()

    return train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, trig_losses_final, excluded_losses_final, models_saved

def train_models_orig(config, runs):
    train, test = gen_train_test(config = config)
    train_losses = torch.zeros(runs, N_EPOCHS)
    test_losses = torch.zeros(runs, N_EPOCHS)
    train_accuracies = torch.zeros(runs, N_EPOCHS)
    test_accuracies = torch.zeros(runs, N_EPOCHS)
    models_saved = []
    for run in tqdm(range(runs)):
        model = Transformer(config, use_cache=False)
        model.to(config.device)
        optimizer = optim.AdamW(model.parameters(), lr = config.lr, weight_decay=config.weight_decay, betas=(0.9, 0.98))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step/10, 1)) # TODO make this a config option
        for epoch in tqdm(range(N_EPOCHS)):
            train_loss, test_loss, train_accuracy, test_accuracy = do_a_training_step(config, model, train, test, optimizer, scheduler, epoch)
            train_losses[run, epoch] = train_loss
            test_losses[run, epoch] = test_loss
            train_accuracies[run, epoch] = train_accuracy
            test_accuracies[run, epoch] = test_accuracy
            #models_saved += [copy.deepcopy(model)]
            if epoch % 100 == 0:
              print(
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Loss: {train_loss}, Test Loss: {test_loss}", '\n',
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}"
              )
    train_losses_final = train_losses.mean(dim=0)
    test_losses_final = test_losses.mean(dim=0)
    train_accuracies_final = train_accuracies.mean(dim=0)
    test_accuracies_final = test_accuracies.mean(dim=0)
    torch.cuda.empty_cache()
    
    return train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved

torch.cuda.empty_cache()
runs = 1
train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, trig_losses_final, excluded_losses_final, models_saved = train_models(config, train_loader, test_loader, all_data, labels, runs)
#train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved = train_models_orig(config, runs)

In [ ]:
from devinterp.slt import estimate_learning_coeff_with_summary

def estimate_rlcts(models, train_loader, criterion, data_length, device, num_draws, num_models):
    estimates = {"sgnht": [], "sgld": []}
    for idx, model in enumerate(tqdm(models)):
        for method, optimizer_kwargs in [
            #("sgnht", {"lr": 1e-7, "diffusion_factor": 0.01}),
            ("sgld", {"lr": 1e-5, "localization": 100.0, "noise_level": 1.0}),
        ]:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                criterion=criterion,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=1,
                num_draws=num_draws,
                num_burnin_steps=100,
                num_steps_bw_draws=1,
                device=device,
                seed=0
            )
            estimate = results["llc/mean"]
            estimates[method].append(estimate)
    return estimates

def obtain_rlct_estimates(train_loader, models_saved, criterion, runs):
    #num_models = 100 + (N_EPOCHS - 1000) // 100 if N_EPOCHS > 1000 else N_EPOCHS // 10
    num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS
    data_length = len(train_loader)
    rlct_estimates = {"sgnht": torch.zeros(runs, num_models), "sgld": torch.zeros(runs, num_models)}
    num_draws = 400

    for run in tqdm(range(runs)):
        rlct_estimate = estimate_rlcts(
            models_saved[num_models * run : num_models * (run + 1)], train_loader, criterion, data_length, DEVICE, num_draws, num_models
        )
        #rlct_estimates["sgnht"][run] = torch.tensor(rlct_estimate["sgnht"])
        rlct_estimates["sgld"][run] = torch.tensor(rlct_estimate["sgld"])

    rlct_estimates_final = {"sgnht": rlct_estimates["sgnht"].mean(dim=0), "sgld": rlct_estimates["sgld"].mean(dim=0)}
    return rlct_estimates_final

#rlct_estimates_final = obtain_rlct_estimates(train_loader, models_saved, criterion, runs)

In [ ]:
dataset = 25

def plot_losses(train_losses_final, test_losses_final, other_loss, name_other_loss, dataset):

    sns.set_style("whitegrid")
    # more frequent checkpoints within first 1000 epochs
    #first_part = np.arange(1, 1001, 10)
    # less frequent thereafter
    #second_part = np.arange(1001, N_EPOCHS, 100)
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_losses_final, label="Train Loss, sgd", color=PRIMARY)
    ax1.plot(x_axis, test_losses_final, label="Test Loss, sgd", color=PRIMARY_LIGHT)
    ax1.plot(x_axis, other_loss.detach(), label=name_other_loss, color=SECONDARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_" + str(dataset) + "_" + name_other_loss + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_accuracies(train_accuracies_final, test_accuracies_final, dataset):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_accuracies_final, label="Train Accuracy, sgd", color=PRIMARY)
    ax1.plot(x_axis, test_accuracies_final, label="Test Accuracy, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_rlcts(rlct_estimates_final, dataset, rlct_estimates_final_other = {}):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    if rlct_estimates_final_other:
        ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="summed curve", color=TERTIARY)
    ax2.plot(x_axis, rlct_estimates_final["sgld"], label="SGLD, sgd", color=TERTIARY_LIGHT)
    ax2.axvline(x=1400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=9400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=14000, color='r', linestyle='--', linewidth=1)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_" + dataset + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_rlcts_circuits(rlct_estimates_final_gen, rlct_estimates_final_mem, rlct_estimates_final_other):

    sns.set_style("whitegrid")
    
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    ax2.plot(x_axis, rlct_estimates_final_gen["sgld"], label="Gen", color=PRIMARY_LIGHT)
    ax2.plot(x_axis, rlct_estimates_final_mem["sgld"], label="Mem", color=TERTIARY)
    ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="Other", color=QUATERNARY)
    ax2.axvline(x=1400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=9400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=14000, color='r', linestyle='--', linewidth=1)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_circuits_" + str(N_EPOCHS) + "_epochs.png")

def plot_losses_chain(last_chain_losses_final, dataset):
    sns.set_style("whitegrid")

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Draw")
    ax1.set_ylabel("Loss", color=PRIMARY)
    ax1.plot(last_chain_losses_final, label="Loss, sgd", color=PRIMARY)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("last_chain_losses_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

plot_losses(train_losses_final, test_losses_final, trig_losses_final, 'Restricted_Loss', dataset)
plot_losses(train_losses_final, test_losses_final, excluded_losses_final, 'Excluded_Loss', dataset)
plot_accuracies(train_accuracies_final, test_accuracies_final, dataset)
plot_rlcts(rlct_estimates_final, dataset='full')
#plot_losses_chain(last_chain_losses_final, dataset)

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS

def d_dt(steps, values):
    slope = np.zeros(len(steps))

    # Compute Slope and Curvature
    for i in range(1, len(steps) - 1):
        dx1 = steps[i+1] - steps[i]
        dx0 = steps[i] - steps[i-1]
        
        dy1 = values[i+1] - values[i]
        dy0 = values[i] - values[i-1]
        
        slope[i] = (dy1 / dx1 + dy0 / dx0) / 2

    slope[0] = slope[1]
    slope[-1] = slope[-2]

    return slope
'''
first_part = np.arange(1, 1001, 10)
    
# Create array from 1000 to 50000 with step 100
# Start from 1100 to avoid duplicating 1000
second_part = np.arange(1001, N_EPOCHS, 100)
    
# Combine the two arrays
x_axis = np.concatenate([first_part, second_part])
'''
x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)
_steps = x_axis.reshape((-1, 1))
_y = rlct_estimates_final['sgld']

kernel = C(1.0, (1e-3, 1e3)) * RBF(3, (5e-1, 1e3))

# Create a Gaussian Process Regressor
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)

# Fit the Gaussian Process
gp.fit(_steps, _y)
_ypred = gp.predict(_steps)
_derivy = d_dt(_steps, _ypred)
print(x_axis[np.abs(_derivy) < .25])


sns.set_style("whitegrid")

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel("Epoch")
ax1.set_ylabel("d lambda / dt", color=PRIMARY)
#plt.yscale('log')
ax1.plot(x_axis, _derivy, label="Slope, sgd", color=PRIMARY_LIGHT)
plt.axhline(y=0, color='k', linestyle=':', linewidth=1)
plt.axvline(x=1400, color='r', linestyle='--', linewidth=1)
plt.axvline(x=9400, color='r', linestyle='--', linewidth=1)
plt.axvline(x=14000, color='r', linestyle='--', linewidth=1)
ax1.tick_params(axis="y", labelcolor=PRIMARY)
ax1.legend(loc="upper left")
fig.tight_layout()
plt.show()
fig.savefig("slope_rlct_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

In [ ]:
torch.save(models_saved, 'models_saved_modular_addition.pt')
torch.save(train_losses_final, 'train_losses_final_modular_addition.pt')
torch.save(test_losses_final, 'test_losses_final_modular_addition.pt')
torch.save(train_accuracies_final, 'train_accuracies_final_modular_addition.pt')
torch.save(test_accuracies_final, 'test_accuracies_final_modular_addition.pt')
torch.save(rlct_estimates_final, 'rlct_estimates_final_modular_addition.pt')
torch.save(trig_losses_final, 'trig_losses_final_modular_addition.pt')
torch.save(excluded_losses_final, 'excluded_losses_final_modular_addition.pt')
torch.save(_derivy, 'derivy_modular_addition.pt')

In [ ]:
models_saved = torch.load('models_saved_modular_addition.pt')
train_losses_final = torch.load('train_losses_final_modular_addition.pt')
test_losses_final = torch.load('test_losses_final_modular_addition.pt')
train_accuracies_final = torch.load('train_accuracies_final_modular_addition.pt')
test_accuracies_final = torch.load('test_accuracies_final_modular_addition.pt')
rlct_estimates_final = torch.load('rlct_estimates_final_modular_addition.pt')
trig_losses_final = torch.load('trig_losses_final_modular_addition.pt')
excluded_losses_final = torch.load('excluded_losses_final_modular_addition.pt')
_derivy = torch.load('derivy_modular_addition.pt')
trig_loss_diffs = torch.load('trig_loss_diffs_modular_addition.pt')
excluded_loss_diffs = torch.load('excluded_loss_diffs_modular_addition.pt')

In [ ]:
rlct_estimates_final = torch.load('rlct_estimates_final.pt')

In [ ]:
def random_partition_of_weights(tensor):
    tensor_shape = tensor.shape

    num_elements = tensor.numel()

    # Step 2: Generate shuffled indices
    indices = torch.randperm(num_elements)

    # Step 3: Split the indices into three disjoint sets
    split_1 = num_elements // 3
    split_2 = 2 * split_1

    indices_A = indices[:split_1]
    indices_B = indices[split_1:split_2]
    indices_C = indices[split_2:]

    # Step 4: Create three masks
    mask_A = torch.zeros(num_elements, dtype=torch.bool)
    mask_B = torch.zeros(num_elements, dtype=torch.bool)
    mask_C = torch.zeros(num_elements, dtype=torch.bool)

    mask_A[indices_A] = True
    mask_B[indices_B] = True
    mask_C[indices_C] = True

    # Step 5: Reshape the masks to the original tensor shape
    mask_A = mask_A.view(tensor_shape)
    mask_B = mask_B.view(tensor_shape)
    mask_C = mask_C.view(tensor_shape)
    
    return mask_A, mask_B, mask_C

def return_topk_percent_mask(tensor, proportion):
    # Step 1: Flatten the tensor
    flattened_tensor = tensor.flatten()

    # Step 2: Determine K, where K is 20% of the total number of elements
    total_elements = flattened_tensor.numel()
    K = int(proportion * total_elements)

    # Step 3: Find the value of the K-th largest element
    topk_values, _ = torch.topk(flattened_tensor, K)
    threshold_value = topk_values[-1]

    # Step 4: Create a boolean mask of the top K values
    return tensor >= threshold_value


def unravel_index(index, shape):
    out = []
    for dim in reversed(shape):
        out.append(index % dim)
        index = index // dim
    return tuple(reversed(out))

def ablation_study(model, loss_fn, config):
    
    trig_loss_diffs = {}
    excluded_loss_diffs = {}
    
    for name, param in model.named_parameters():
        trig_loss_diffs[name] = torch.zeros(param.shape)
        excluded_loss_diffs[name] = torch.zeros(param.shape)

    print(' ablation study begins: ')
    fourier_basis = make_fourier_basis(config = config)
    train, _ = gen_train_test(config = config)
    is_train, is_test = config.is_train_is_test(train = train)
    all_data = torch.tensor([(i, j, config.p) for i in range(config.p) for j in range(config.p)]).to(config.device)
    labels = torch.tensor([config.fn(i, j) for i, j, _ in all_data]).to(config.device)
                
    indices = get_indices_for_key_freqs_calculation(config)
    key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
    logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
    trig_loss_baseline, trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
    excluded_loss_baseline = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
    
    for name, param in model.named_parameters():
        param_shape = param.shape
        for idx in tqdm(range(param.numel()), desc=f'Ablating {name}'):
            with torch.no_grad():
                # Convert flat index i to multi-dimensional index for the original shape
                multi_idx = unravel_index(idx, param_shape)
                
                # Save the original weight value
                original_value = param[multi_idx].item()
                
                # Set the weight to zero
                param[multi_idx] = 0.0
                
                # Compute new metrics
                key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
                logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
                trig_loss, trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
                excluded_loss = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
                
                # Calculate differences
                trig_loss_diff = trig_loss - trig_loss_baseline if trig_loss > trig_loss_baseline else trig_loss_baseline - trig_loss
                excluded_loss_diff = excluded_loss - excluded_loss_baseline if excluded_loss > excluded_loss_baseline else excluded_loss_baseline - excluded_loss
                
                trig_loss_diffs[name][multi_idx] = trig_loss_diff
                excluded_loss_diffs[name][multi_idx] = excluded_loss_diff
            
                # Restore the original weight
                param[multi_idx] = original_value
                torch.cuda.empty_cache()
    
    return trig_loss_diffs, excluded_loss_diffs

# Use the function
#trig_loss_diffs, excluded_loss_diffs = ablation_study(models_saved[-1], criterion, config)
#torch.save(trig_loss_diffs, 'trig_loss_diffs.pt')
#torch.save(excluded_loss_diffs, 'excluded_loss_diffs.pt')
trig_loss_diffs = torch.load('trig_loss_diffs.pt')
excluded_loss_diffs = torch.load('excluded_loss_diffs.pt')

gen_circuit_weights = 0
mem_circuit_weights = 0
overlap = 0

gen_model_indices = {}
mem_model_indices = {}
other_model_indices = {}

# Analyze results
for name in tqdm(trig_loss_diffs.keys()):
    print(f"Layer: {name}")
    '''
    proportion = 0.3
    trig_indices = return_topk_percent_mask(trig_loss_diffs[name], proportion)
    exc_indices = return_topk_percent_mask(excluded_loss_diffs[name], proportion + .2)
    both = trig_indices & exc_indices
    
    gen_model_indices[name] = trig_indices
    mem_model_indices[name] = exc_indices & ~both
    other_model_indices[name] = ~both
    
    above_eps_trig = trig_loss_diffs[name][gen_model_indices[name]]
    above_eps_excluded = excluded_loss_diffs[name][mem_model_indices[name]]
    present_in_both = trig_loss_diffs[name][both]
    
    #print('present in both: ', torch.count_nonzero(both).item())
    gen_circuit_weights += above_eps_trig.shape[0]
    mem_circuit_weights += above_eps_excluded.shape[0]
    overlap += present_in_both.shape[0]
    
    print((gen_model_indices[name] | mem_model_indices[name] | other_model_indices[name]).all())
    '''
    maskA, maskB, maskC = random_partition_of_weights(trig_loss_diffs[name])
    
    gen_model_indices[name] = maskA
    mem_model_indices[name] = maskB
    other_model_indices[name] = maskC
    
print('number of gen circuit weights: ', gen_circuit_weights)
print('number of mem circuit weights: ', mem_circuit_weights)
print('present in both: ', overlap)
print(sum(p.numel() for p in models_saved[-1].parameters()))

'''
topk_trig_loss_diffs, indices1 = torch.topk(np.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(np.abs(excluded_loss_diffs), k)
A = weight_ids[indices1]
B = weight_ids[indices2]
print(np.intersect1d(A, B))
'''

In [ ]:
k = 10000

print('number of gen circuit weights: ', gen_circuit_weights)
print('number of mem circuit weights: ', mem_circuit_weights)
topk_trig_loss_diffs, indices1 = torch.topk(np.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(np.abs(excluded_loss_diffs), k)
A = weight_ids[indices1]
B = weight_ids[indices2]
intersection = torch.tensor([x for x in A if x in B])
print(len(intersection), k - len(intersection))

In [ ]:
k = 1000

# Get top k elements from both lists
topk_trig_loss_diffs, indices1 = torch.topk(torch.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(torch.abs(excluded_loss_diffs), k)

# Get the corresponding weight ids
A = weight_ids[indices1]
B = weight_ids[indices2]

# Find unique elements in A that are not in B and vice versa
unique_to_A = torch.tensor([x for x in A if x not in B])
unique_to_B = torch.tensor([x for x in B if x not in A])

# Find intersection
intersection = torch.tensor([x for x in A if x in B])

print("Intersection:", len(intersection))
print("Unique to A:", len(unique_to_A))
print("Unique to B:", len(unique_to_B))
print(unique_to_A)
print(unique_to_B)

In [ ]:
import copy

def prune_to_obtain_circuit(gen_model, gen_model_indices, mem_model, mem_model_indices, other_model, other_model_indices):
    
    gen_state_dict = gen_model.state_dict()
    mem_state_dict = mem_model.state_dict()
    other_state_dict = other_model.state_dict()
    
    for name, param in gen_model.named_parameters():
        gen_indices = gen_model_indices[name]
        gen_state_dict[name][~gen_indices] = 0.0
        
        mem_indices = mem_model_indices[name]
        mem_state_dict[name][~mem_indices] = 0.0
        
        other_indices = other_model_indices[name]
        other_state_dict[name][~other_indices] = 0.0
        
    gen_model.load_state_dict(gen_state_dict)
    mem_model.load_state_dict(mem_state_dict)
    other_model.load_state_dict(other_state_dict)
            
    return gen_model, mem_model, other_model

gen_models = []
mem_models = []
other_models = []

for model in tqdm(models_saved):
    gen_model = copy.deepcopy(model)
    mem_model = copy.deepcopy(model)
    other_model = copy.deepcopy(model)
    
    gen_model, mem_model, other_model = prune_to_obtain_circuit(gen_model, gen_model_indices, mem_model, mem_model_indices, other_model, other_model_indices)
    
    gen_models.append(gen_model)
    mem_models.append(mem_model)
    other_models.append(other_model)
    
torch.save(gen_models, 'gen_models.pt')
torch.save(mem_models, 'mem_models.pt')
torch.save(other_models, 'other_models.pt')

In [ ]:
gen_models = torch.load('gen_models.pt')
mem_models = torch.load('mem_models.pt')
other_models = torch.load('other_models.pt')

In [ ]:
rlct_estimates_final_gen = obtain_rlct_estimates(train_loader, gen_models, criterion, runs)
torch.save(rlct_estimates_final_gen, 'rlct_estimates_final_gen.pt')

In [ ]:
rlct_estimates_final_mem = obtain_rlct_estimates(train_loader, mem_models, criterion, runs)
torch.save(rlct_estimates_final_mem, 'rlct_estimates_final_mem.pt')

In [ ]:
rlct_estimates_final_other = obtain_rlct_estimates(train_loader, other_models, criterion, runs)
torch.save(rlct_estimates_final_other, 'rlct_estimates_final_other.pt')

In [ ]:
rlct_estimates_final_gen = torch.load('rlct_estimates_final_gen.pt')
rlct_estimates_final_mem = torch.load('rlct_estimates_final_mem.pt')
rlct_estimates_final_other = torch.load('rlct_estimates_final_other.pt')

In [ ]:
plot_rlcts_circuits(rlct_estimates_final_gen, rlct_estimates_final_mem, rlct_estimates_final_other)
summed_curves = {}
summed_curves['sgld'] = rlct_estimates_final_gen['sgld'] + rlct_estimates_final_mem['sgld'] + rlct_estimates_final_other['sgld']
plot_rlcts(rlct_estimates_final, dataset='comparison_with_summed_curves', rlct_estimates_final_other=summed_curves)
plot_rlcts(rlct_estimates_final_gen, dataset='gen')
plot_rlcts(rlct_estimates_final_mem, dataset='mem')
plot_rlcts(rlct_estimates_final_other, dataset='other')

In [ ]:
print(rlct_estimates_final_gen['sgld'])
print(rlct_estimates_final_mem['sgld'])
print(rlct_estimates_final_other['sgld'])

In [ ]:
print(all_data.shape)

for name, param in models_saved[-1].named_parameters():
    print(name)
    print(param.shape)